# 3. Multi-Model Training & Evaluation

This notebook trains multiple machine learning classifiers on the SMOTENC-resampled training data (Experiment 3 strategy), performs **Stratified 6-Fold Cross-Validation** using the optimal threshold of **0.70**, and evaluates them on the untouched held-out test split.  

The candidate models evaluated are:
1. **Logistic Regression**
2. **Decision Tree**
3. **Random Forest**
4. **XGBoost**

## 1. Imports & Setup

In [1]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print('Imports complete.')

Imports complete.


## 2. Load Prepared Dataset

In [2]:
BASE_DIR = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'artifacts').exists() or (candidate / 'requirements.txt').exists():
        BASE_DIR = candidate
        break
if BASE_DIR is None:
    BASE_DIR = Path.cwd()

ARTIFACT_DIR = BASE_DIR / 'artifacts' / 'data'
EVAL_DIR     = BASE_DIR / 'artifacts' / 'evaluation'
MODEL_DIR    = BASE_DIR / 'artifacts' / 'models'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_train.npz')['X_train']
X_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_test.npz')['X_test']
y_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_train.npz')['y_train']
y_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_test.npz')['y_test']

with open(ARTIFACT_DIR / 'features.json', 'r') as f:
    feature_names = json.load(f)

with open(ARTIFACT_DIR / 'threshold.json', 'r') as f:
    OPTIMAL_THRESHOLD = json.load(f)['optimal_threshold']

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

print(f'X_train shape: {X_train.shape}  (fraud count: {int(y_train.sum()):,})')
print(f'X_test shape : {X_test.shape}   (fraud count: {int(y_test.sum()):,})')
print(f'Optimal threshold: {OPTIMAL_THRESHOLD}')

X_train shape: (1134468, 29)  (fraud count: 103,133)
X_test shape : (259335, 29)   (fraud count: 1,501)
Optimal threshold: 0.7


## 3. Define Candidate Models

In [3]:
models = {
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED, max_depth=8),
    'Random Forest': RandomForestClassifier(random_state=SEED, n_estimators=100, max_depth=8, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=SEED, eval_metric='logloss', n_estimators=100, max_depth=6, n_jobs=-1)
}
print(f'Defined {len(models)} candidate models.')

Defined 4 candidate models.


## 4. Stratified 6-Fold Cross-Validation

Train each model using cross-validation. Apply the optimal threshold of **0.70** to validation predictions, select the fold estimator that achieves the highest validation F1 score, and track validation averages.

In [4]:
cv = StratifiedKFold(n_splits=6, random_state=SEED, shuffle=True)
trained_models = {}
cv_summary = []

for model_name, base_model in models.items():
    print(f'\nRunning K-Fold CV for {model_name}...')
    fold_f1s = []
    fold_estimators = []
    fold_metrics = []

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_df, y_train), start=1):
        X_fold_train, X_fold_val = X_train_df.iloc[train_idx], X_train_df.iloc[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

        # Clone and train
        from sklearn.base import clone
        fold_model = clone(base_model)
        fold_model.fit(X_fold_train, y_fold_train)

        # Validation probabilities
        y_proba_val = fold_model.predict_proba(X_fold_val)[:, 1]
        y_pred_val = (y_proba_val >= OPTIMAL_THRESHOLD).astype(int)

        # Metrics
        f1 = f1_score(y_fold_val, y_pred_val, zero_division=0)
        prec = precision_score(y_fold_val, y_pred_val, zero_division=0)
        rec = recall_score(y_fold_val, y_pred_val, zero_division=0)
        auc = roc_auc_score(y_fold_val, y_proba_val)

        fold_f1s.append(f1)
        fold_estimators.append(fold_model)
        fold_metrics.append([prec, rec, f1, auc])

    # Save the best estimator of this type based on validation F1
    best_fold_idx = np.argmax(fold_f1s)
    best_estimator = fold_estimators[best_fold_idx]
    trained_models[model_name] = best_estimator

    # CV Averages
    mean_metrics = np.mean(fold_metrics, axis=0)
    cv_summary.append({
        'Model': model_name,
        'CV Precision': mean_metrics[0],
        'CV Recall': mean_metrics[1],
        'CV F1-Score': mean_metrics[2],
        'CV ROC-AUC': mean_metrics[3]
    })

    print(f'Finished {model_name}. Mean CV F1-Score: {mean_metrics[2]:.4f} (Best Fold F1: {fold_f1s[best_fold_idx]:.4f})')


Running K-Fold CV for Logistic Regression...
Finished Logistic Regression. Mean CV F1-Score: 0.6397 (Best Fold F1: 0.6417)

Running K-Fold CV for Decision Tree...
Finished Decision Tree. Mean CV F1-Score: 0.9199 (Best Fold F1: 0.9223)

Running K-Fold CV for Random Forest...
Finished Random Forest. Mean CV F1-Score: 0.8321 (Best Fold F1: 0.8357)

Running K-Fold CV for XGBoost...
Finished XGBoost. Mean CV F1-Score: 0.9763 (Best Fold F1: 0.9775)


## 5. Evaluate on Held-Out Test Split

Evaluate each model's best estimator on the untouched test split (applying 0.70 threshold).

In [5]:
test_summary = []
test_cms = {}

for model_name, estimator in trained_models.items():
    # Predict
    y_proba_test = estimator.predict_proba(X_test_df)[:, 1]
    y_pred_test = (y_proba_test >= OPTIMAL_THRESHOLD).astype(int)

    # Metrics
    test_summary.append({
        'Model': model_name,
        'Test Accuracy': accuracy_score(y_test, y_pred_test),
        'Test Precision': precision_score(y_test, y_pred_test, zero_division=0),
        'Test Recall': recall_score(y_test, y_pred_test, zero_division=0),
        'Test F1-Score': f1_score(y_test, y_pred_test, zero_division=0),
        'Test ROC-AUC': roc_auc_score(y_test, y_proba_test)
    })
    test_cms[model_name] = confusion_matrix(y_test, y_pred_test)

## 6. Model Comparison Summary

In [6]:
cv_df = pd.DataFrame(cv_summary).set_index('Model')
test_df = pd.DataFrame(test_summary).set_index('Model')
summary_df = cv_df.join(test_df)

print('=== Multi-Model Comparison Summary (threshold = 0.70) ===')
print(summary_df.round(4).to_string())

# Find best model by Test F1
best_model_name = test_df['Test F1-Score'].idxmax()
print(f'\nBest performing model on Test Set: {best_model_name} (F1 = {test_df.loc[best_model_name, "Test F1-Score"]:.4f})')

=== Multi-Model Comparison Summary (threshold = 0.70) ===
                     CV Precision  CV Recall  CV F1-Score  CV ROC-AUC  Test Accuracy  Test Precision  Test Recall  Test F1-Score  Test ROC-AUC
Model                                                                                                                                         
Logistic Regression        0.9456     0.4833       0.6397      0.9658         0.9939          0.4674       0.4250         0.4452        0.9267
Decision Tree              0.9692     0.8754       0.9199      0.9952         0.9961          0.6258       0.8201         0.7099        0.9809
Random Forest              0.9889     0.7182       0.8321      0.9961         0.9971          0.8160       0.6442         0.7200        0.9893
XGBoost                    0.9906     0.9623       0.9763      0.9997         0.9977          0.8095       0.7928         0.8011        0.9979

Best performing model on Test Set: XGBoost (F1 = 0.8011)


## 7. Visualize Confusion Matrices

Plot confusion matrices side-by-side in a 2x2 grid and save to artifacts.

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (model_name, cm) in enumerate(test_cms.items()):
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        xticklabels=['Legitimate', 'Fraud'],
        yticklabels=['Legitimate', 'Fraud']
    )
    axes[idx].set_title(f'{model_name} (threshold={OPTIMAL_THRESHOLD})')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices Comparison — Test Split', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'multi_model_confusion_matrices.png', dpi=150)
plt.show()
print('Comparison plot saved to artifacts/evaluation/multi_model_confusion_matrices.png')

Comparison plot saved to artifacts/evaluation/multi_model_confusion_matrices.png


## 8. Persist All Trained Models

Save the best estimators of all model types to the model artifacts directory.

In [8]:
for model_name, estimator in trained_models.items():
    sanitized_name = model_name.lower().replace(' ', '_')
    save_path = MODEL_DIR / f'{sanitized_name}_model.pkl'
    joblib.dump(estimator, save_path)
    print(f'Saved {model_name} model to: {save_path}')

Saved Logistic Regression model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\logistic_regression_model.pkl
Saved Decision Tree model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\decision_tree_model.pkl
Saved Random Forest model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\random_forest_model.pkl
Saved XGBoost model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\xgboost_model.pkl
